# Daily Challenge: Text Summarization using NLP
This notebook demonstrates how to preprocess text, use GloVe embeddings, compute sentence similarities, and apply PageRank for extractive summarization.

In [ ]:
# 1. Data Loading and Inspection
import pandas as pd
df = pd.read_excel('tennis_articles.xls')
print(df.head())
print(df.info())
df = df.drop('article_title', axis=1)

In [ ]:
# 2. Sentence Tokenization
import nltk
nltk.download('punkt')
sentences = df['article_text'].apply(nltk.sent_tokenize).tolist()
flat_sentences = [sent for sublist in sentences for sent in sublist]
print('Total sentences:', len(flat_sentences))

In [ ]:
# 3. Download and Load GloVe Word Embeddings
import zipfile
import requests
import os
glove_url = 'http://nlp.stanford.edu/data/glove.6B.zip'
glove_path = 'glove.6B.100d.txt'
if not os.path.exists(glove_path):
    r = requests.get(glove_url)
    with open('glove.6B.zip', 'wb') as f:
        f.write(r.content)
    with zipfile.ZipFile('glove.6B.zip', 'r') as zip_ref:
        zip_ref.extract(glove_path)
embeddings = {}
with open(glove_path, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = list(map(float, values[1:]))
        embeddings[word] = vector
print('Loaded GloVe embeddings:', len(embeddings))

In [ ]:
# 4. Text Cleaning and Normalization
import re
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
def clean_sentence(sentence):
    sentence = re.sub(r'[^a-zA-Z ]', '', sentence)
    sentence = sentence.lower()
    words = sentence.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)
cleaned_sentences = [clean_sentence(s) for s in flat_sentences]

In [ ]:
# 5. Sentence Vectorization
import numpy as np
def sentence_vector(sentence, embeddings, dim=100):
    words = sentence.split()
    if len(words) == 0:
        return np.zeros(dim)
    vectors = [embeddings.get(w, np.zeros(dim)) for w in words]
    return np.mean(vectors, axis=0)
sentence_vectors = [sentence_vector(s, embeddings) for s in cleaned_sentences]

In [ ]:
# 6. Similarity Matrix Construction
from sklearn.metrics.pairwise import cosine_similarity
n = len(sentence_vectors)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            sim_matrix[i][j] = cosine_similarity([sentence_vectors[i]], [sentence_vectors[j]])[0,0]

In [ ]:
# 7. Graph Construction and Sentence Ranking
import networkx as nx
nx_graph = nx.from_numpy_array(sim_matrix)
scores = nx.pagerank(nx_graph)

In [ ]:
# 8. Summarization
ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(flat_sentences)), reverse=True)
summary = [s for _, s in ranked_sentences[:10]]
print('Summary:')
for sent in summary:
    print('-', sent)